A. Studi Kasus: Pipeline Prediksi Risiko Gagal Bayar Pinjaman FinTech (Credit Scoring)
Latar Belakang & Masalah: Lembaga pinjaman daring menghadapi risiko nasabah gagal melunasi tagihan (credit default). Dalam industri finansial, kesalahan melepas peminjam macet (False Negative) jauh lebih berbiaya tinggi dibanding salah menolak peminjam lancar.
Solusi:
Membangun pipeline modular Scikit-Learn: ColumnTransformer untuk standarisasi fitur numerik (StandardScaler) dan enkoding fitur kategorikal status kerja (OneHotEncoder).
Menggunakan algoritma Random Forest Classifier dengan pembobotan kelas seimbang (class_weight='balanced') untuk mengatasi ketidakseimbangan kelas (class imbalance).
Menguji model pada data pengujian independen (stratified test set) dan mengukur kurva ROC-AUC.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

def studi_kasus_4_machine_learning():
    np.random.seed(42)
    n = 1200

    # 1. Dataset Sintetis Peminjam Kredit
    age = np.random.randint(21, 65, size=n)
    income = np.random.normal(12_000_000, 4_000_000, size=n).clip(3_000_000, 30_000_000)
    debt_ratio = np.random.beta(2, 5, size=n) # Rasio hutang thd penghasilan (0 s/d 1)
    credit_score = np.random.normal(600, 90, size=n).clip(300, 850)
    job_status = np.random.choice(['Permanent', 'Contract', 'Freelance'], size=n, p=[0.5, 0.35, 0.15])

    # Logika probabilistik default: hutang tinggi + credit score rendah
    logit = -2.2 + 3.8 * debt_ratio - 0.007 * (credit_score - 500) + (job_status == 'Freelance') * 0.9
    prob_default = 1 / (1 + np.exp(-logit))
    default_target = (np.random.rand(n) < prob_default).astype(int)

    df = pd.DataFrame({
        'Age': age, 'Income': income, 'Debt_Ratio': debt_ratio,
        'Credit_Score': credit_score, 'Job_Status': job_status,
        'Default': default_target
    })

    X = df.drop(columns=['Default'])
    y = df['Default']

    # 2. Data Splitting (Stratified agar proporsi default terjaga)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    # 3. Preprocessing Pipeline
    num_cols = ['Age', 'Income', 'Debt_Ratio', 'Credit_Score']
    cat_cols = ['Job_Status']
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols)
    ])

    # 4. Model Training dengan Bobot Berimbang (Balanced Weight)
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=42))
    ])
    pipeline.fit(X_train, y_train)

    # 5. Evaluasi Prediksi
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    print("=== EVALUASI END-TO-END MACHINE LEARNING ===")
    print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    print("\nLaporan Klasifikasi:")
    print(classification_report(y_test, y_pred, target_names=['Lancar (0)', 'Default (1)'], zero_division=0))
    print("Confusion Matrix (Baris: Aktual, Kolom: Prediksi):")
    print(confusion_matrix(y_test, y_pred))

if __name__ == '__main__':
    studi_kasus_4_machine_learning()

=== EVALUASI END-TO-END MACHINE LEARNING ===
ROC-AUC Score: 0.7078

Laporan Klasifikasi:
              precision    recall  f1-score   support

  Lancar (0)       0.86      0.84      0.85       244
 Default (1)       0.38      0.41      0.39        56

    accuracy                           0.76       300
   macro avg       0.62      0.63      0.62       300
weighted avg       0.77      0.76      0.77       300

Confusion Matrix (Baris: Aktual, Kolom: Prediksi):
[[206  38]
 [ 33  23]]
